# Практика 3. Линейные модели и градиентный спуск — версия с решениями

> Этот файл предназначен преподавателю: все задания заполнены, добавлены комментарии к разбору. Студентам выдаётся версия без суффикса `_solutions`.

# Практика 3. Линейные модели и градиентный спуск

**Курс «Машинное обучение» · Часть 1, занятие 3**

Сегодня пишем градиентный спуск руками. Это упражнение, которое стоит проделать один раз в жизни: после него становится понятно, что происходит внутри `fit` — и у линейной модели, и у нейросети.

План:

1. Градиентный спуск с нуля: batch, стохастический, мини-пакетный
2. Скорость обучения и что бывает, когда её выбрали неудачно
3. Регуляризация: Ridge и Lasso
4. Сравнение с реализацией из scikit-learn
5. Логистическая регрессия и её коэффициенты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, SGDRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("готово")

> **Что берём авансом.** Кросс-валидация и метрики R² и ROC-AUC —
> тема четвёртой лекции. Сегодня они нужны как способ сравнить два
> варианта обучения между собой; подробный разбор, что эти числа значат
> и когда врут, будет там.

---
## 1. Данные

Берём встроенный датасет о прогрессировании диабета: 442 пациента, 10 признаков, целевая переменная — количественная мера развития болезни через год. Данные уже отмасштабированы авторами.

In [ ]:
data = load_diabetes(as_frame=True)
X_df, y_s = data.data, data.target

print(f"объектов: {X_df.shape[0]}, признаков: {X_df.shape[1]}")
print(f"целевая: от {y_s.min():.0f} до {y_s.max():.0f}, среднее {y_s.mean():.1f}")
display(X_df.head(3))

X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y_s, test_size=0.2, random_state=RANDOM_STATE
)

# Для собственных реализаций работаем с numpy-массивами
scaler = StandardScaler().fit(X_train_df)
X_train = scaler.transform(X_train_df)
X_test = scaler.transform(X_test_df)
y_train_np = y_train.to_numpy()
y_test_np = y_test.to_numpy()

print(f"\nобучающая: {X_train.shape}, тестовая: {X_test.shape}")

---
## 2. Градиентный спуск своими руками

Напомню формулы с лекции. Предсказание, функция потерь и градиент:

$$\hat{y} = Xw + b, \qquad L = \frac{1}{n}\|y - \hat{y}\|^2$$

$$\frac{\partial L}{\partial w} = -\frac{2}{n}X^T(y - \hat{y}), \qquad \frac{\partial L}{\partial b} = -\frac{2}{n}\sum(y - \hat{y})$$

Свободный член держим отдельно от весов: так его удобнее исключать из регуляризации.

In [ ]:
class LinearRegressionGD:
    """Линейная регрессия, обучаемая градиентным спуском.

    mode:
        'batch' — полный градиент по всей выборке
        'sgd'   — по одному случайному объекту
        'mini'  — по мини-пакету

    penalty: None, 'l2' или 'l1'
    """

    def __init__(self, lr=0.1, n_epochs=100, mode="batch", batch_size=32,
                 penalty=None, alpha=0.0, random_state=42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.mode = mode
        self.batch_size = batch_size
        self.penalty = penalty
        self.alpha = alpha
        self.random_state = random_state

    def _grad_penalty(self):
        """Производная штрафа по весам. Свободный член не штрафуем."""
        if self.penalty == "l2":
            return 2 * self.alpha * self.w
        if self.penalty == "l1":
            return self.alpha * np.sign(self.w)
        return 0.0

    def _loss(self, X, y):
        residual = y - (X @ self.w + self.b)
        loss = np.mean(residual ** 2)
        if self.penalty == "l2":
            loss += self.alpha * np.sum(self.w ** 2)
        elif self.penalty == "l1":
            loss += self.alpha * np.sum(np.abs(self.w))
        return loss

    def fit(self, X, y):
        rng = np.random.default_rng(self.random_state)
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        # Первая точка истории — ошибка ДО обучения. Без неё кривые разных
        # режимов начинаются с разной высоты: за одну эпоху sgd успевает
        # сделать столько шагов, сколько в выборке объектов, а batch — один
        self.history_ = [self._loss(X, y)]

        for epoch in range(self.n_epochs):
            if self.mode == "batch":
                batches = [np.arange(n)]
            else:
                idx = rng.permutation(n)
                size = 1 if self.mode == "sgd" else self.batch_size
                batches = [idx[i:i + size] for i in range(0, n, size)]

            for batch in batches:
                Xb, yb = X[batch], y[batch]
                residual = yb - (Xb @ self.w + self.b)
                grad_w = -2 / len(batch) * (Xb.T @ residual) + self._grad_penalty()
                grad_b = -2 / len(batch) * residual.sum()
                self.w -= self.lr * grad_w
                self.b -= self.lr * grad_b

            self.history_.append(self._loss(X, y))
        return self

    def predict(self, X):
        return X @ self.w + self.b


print("класс определён")

In [ ]:
# Проверяем: сходится ли
model_gd = LinearRegressionGD(lr=0.1, n_epochs=200, mode="batch")
model_gd.fit(X_train, y_train_np)

plt.figure(figsize=(9, 4))
plt.plot(model_gd.history_, color="#2B5CE6")
plt.xlabel("эпоха")
plt.ylabel("MSE на обучающей выборке")
plt.title("Кривая обучения")
plt.show()

print(f"MSE в начале: {model_gd.history_[0]:.1f}")
print(f"MSE в конце:  {model_gd.history_[-1]:.1f}")

In [ ]:
# Сверяемся с аналитическим решением из scikit-learn
sk = LinearRegression().fit(X_train, y_train_np)

print("сравнение весов:")
comparison = pd.DataFrame({
    "наш GD": model_gd.w.round(2),
    "sklearn": sk.coef_.round(2),
}, index=X_df.columns)
display(comparison)

print(f"\nмаксимальное расхождение: {np.abs(model_gd.w - sk.coef_).max():.3f}")
print(f"свободный член: наш {model_gd.b:.2f}, sklearn {sk.intercept_:.2f}")

> **Для преподавателя.** Вопрос, который задают каждый год: почему sgd «сразу лучше». Потому что на оси эпохи, а не шаги оптимизации. Стоит предложить студентам мысленно перестроить график по числу обновлений весов — кривые почти совпадут, а преимущество sgd исчезнет.

> **Для преподавателя.** Небольшое расхождение весов — норма: градиентный спуск за 200 эпох не доходит до точного минимума. Если увеличить число эпох, веса сойдутся к аналитическому решению. Полезно показать это студентам прямо на паре.

### Задание 1

Сравните три режима обучения — `batch`, `sgd` и `mini`. Цикл, графики и таблица уже написаны: **допишите две строки** внутри цикла — создайте модель и посчитайте MSE на тесте.

Когда всё заработает, посмотрите на результат и ответьте себе: какой режим сходится быстрее по числу эпох, а какой по времени?

> **Что посмотреть:** у класса `LinearRegressionGD` из ячейки выше есть аргументы `mode` (`"batch"`, `"sgd"`, `"mini"`), `batch_size`, `lr`, `n_epochs`, а история функции потерь лежит в `model.history_`. Время меряют через `time.perf_counter()` до и после, ошибку — через `mean_squared_error(y_true, y_pred)`.

In [ ]:
import time

modes = {
    "batch": dict(mode="batch"),
    "sgd": dict(mode="sgd"),
    "mini (32)": dict(mode="mini", batch_size=32),
}

plt.figure(figsize=(11, 4.5))
results = {}

for name, kwargs in modes.items():
    start = time.perf_counter()

    # Режим и размер пакета приходят из словаря через **kwargs
    model = LinearRegressionGD(lr=0.01, n_epochs=100,
                               random_state=RANDOM_STATE,
                               **kwargs).fit(X_train, y_train_np)
    mse = mean_squared_error(y_test_np, model.predict(X_test))

    elapsed = time.perf_counter() - start
    results[name] = (model.history_[-1], mse, elapsed)
    plt.plot(model.history_, label=name)

plt.xlabel("эпоха (0 — состояние до обучения)")
plt.ylabel("MSE на обучающей выборке")
plt.title("Три режима градиентного спуска")
plt.legend()
plt.show()

print(f"{'режим':12} {'MSE train':>11} {'MSE test':>11} {'время, с':>10}")
for name, (tr, te, t) in results.items():
    print(f"{name:12} {tr:11.1f} {te:11.1f} {t:10.3f}")

mse_batch = results["batch"][1]
mse_sgd = results["sgd"][1]
mse_mini = results["mini (32)"][1]

print()
print("Выводы:")
print("  По числу эпох быстрее всех сходятся sgd и mini: за одну эпоху они")
print("  делают много обновлений весов, а batch — только одно.")
print("  По времени batch выигрывает: одна матричная операция на всю выборку")
print("  эффективнее сотен мелких. На больших данных картина обратная —")
print("  там batch не помещается в память, и мини-пакетный режим побеждает.")
print("  Кривая sgd заметно шумнее: каждый шаг делается по одному объекту.")

print()
n_train = len(X_train)
print("Почему кривые расходятся уже на первой эпохе:")
# history_[0] — ошибка ДО первого шага, она одна и та же у всех режимов
start_loss = model.history_[0]
print(f"  все три стартуют из одной точки: w = 0, ошибка {start_loss:.0f}")
print("  но за ОДНУ эпоху они делают разное число обновлений весов:")
print(f"    batch     — 1 обновление")
print(f"    mini (32) — {int(np.ceil(n_train / 32))} обновлений")
print(f"    sgd       — {n_train} обновлений, по одному на каждый объект")
print("  Эпоха считает проходы по данным, а не шаги оптимизации. По числу")
print("  шагов batch не хуже — его градиент точный, а не оценка по одному объекту.")


In [ ]:
# Проверка задания 1
assert round(float(mse_batch)) == 3431, f"mse_batch = {mse_batch}"
assert round(float(mse_sgd)) == 3423, f"mse_sgd = {mse_sgd}"
assert round(float(mse_mini)) == 2832, f"mse_mini = {mse_mini}"
print("Задание 1 — верно ✓")

In [ ]:
# Что происходит с весами по ходу обучения
model_track = LinearRegressionGD(lr=0.05, n_epochs=1)
model_track.w = np.zeros(X_train.shape[1])
model_track.b = 0.0

weight_history = []
for epoch in range(200):
    resid = y_train_np - (X_train @ model_track.w + model_track.b)
    grad_w = -2 / len(X_train) * (X_train.T @ resid)
    grad_b = -2 / len(X_train) * resid.sum()
    model_track.w -= 0.05 * grad_w
    model_track.b -= 0.05 * grad_b
    weight_history.append(model_track.w.copy())

weight_history = np.array(weight_history)

plt.figure(figsize=(11, 5))
for j, name in enumerate(X_df.columns):
    plt.plot(weight_history[:, j], label=name, lw=1.6)
plt.xlabel("эпоха"); plt.ylabel("значение веса")
plt.title("Как веса приходят к своим значениям")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

print("Одни веса устанавливаются за десяток эпох, другие продолжают")
print("двигаться сотню. Это и есть та самая разная кривизна поверхности")
print("по разным направлениям.")

### Вопрос

Некоторые веса сходятся за 10 эпох, некоторые за 150. Как это связано с формой поверхности потерь и что можно сделать, чтобы выровнять скорость сходимости?

**Что хотелось услышать:**

Скорость сходимости по каждому направлению определяется кривизной поверхности вдоль него. Там, где функция меняется резко, шаг нужного размера доходит до минимума быстро. Там, где она почти плоская, тот же шаг продвигает мало — и требуется много итераций.

Выровнять можно тремя способами. Первый — масштабирование признаков: оно делает поверхность более симметричной. Второй — адаптивная скорость обучения: у каждого параметра свой шаг, обратно пропорциональный накопленной величине градиента (так работают AdaGrad, RMSProp, Adam). Третий — момент: он накапливает движение вдоль пологих направлений.

Во второй части курса эти методы разбираются подробно — там без них нейросеть просто не обучится.

---
## 3. Скорость обучения

Главный гиперпараметр метода. Посмотрим, что бывает при разных значениях.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Логарифмическая шкала обязательна: MSE падает с 30 000 до 3 000,
# и на линейной оси все различия между кривыми сплющиваются у дна
for lr in [0.001, 0.01, 0.05, 0.2]:
    m = LinearRegressionGD(lr=lr, n_epochs=100).fit(X_train, y_train_np)
    axes[0].plot(m.history_, lw=2, label=f"lr = {lr}")

axes[0].set_yscale("log")
axes[0].set_xlabel("эпоха")
axes[0].set_ylabel("MSE (логарифмическая шкала)")
axes[0].set_title("Рабочий диапазон: чем крупнее шаг, тем быстрее")
axes[0].legend()

# А теперь слишком большая скорость
m_big = LinearRegressionGD(lr=0.5, n_epochs=30).fit(X_train, y_train_np)
axes[1].plot(m_big.history_, color="#C0393B", lw=2)
axes[1].set_yscale("log")
axes[1].set_xlabel("эпоха")
axes[1].set_ylabel("MSE (логарифмическая шкала)")
axes[1].set_title("lr = 0.5 — расходимость")
plt.tight_layout()
plt.show()

print("Шаг 0.001 за сто эпох не доходит: MSE всё ещё около 20 000.")
print("Шаг 0.2 садится в минимум за пару десятков эпох.")
print(f"MSE при lr = 0.5 на последней эпохе: {m_big.history_[-1]:.3e}")
print("Потери растут вместо того, чтобы убывать: шаги перепрыгивают минимум")
print("и с каждым разом уходят всё дальше.")

---
## 4. Регуляризация

Добавим штраф за величину весов и посмотрим, что он делает.

In [ ]:
# Диапазон alpha ограничен сверху не случайно: в шаге с L2-штрафом
# сидит множитель (1 - 2*lr*alpha), и при alpha больше 1/lr спуск
# расходится. Это тот же урок про скорость обучения, вид сбоку
alphas = np.logspace(-2, 1, 25)
coefs_l2, coefs_l1 = [], []

for a in alphas:
    m2 = LinearRegressionGD(lr=0.05, n_epochs=300, penalty="l2", alpha=a)
    coefs_l2.append(m2.fit(X_train, y_train_np).w.copy())

    m1 = LinearRegressionGD(lr=0.05, n_epochs=300, penalty="l1", alpha=a)
    coefs_l1.append(m1.fit(X_train, y_train_np).w.copy())

coefs_l2 = np.array(coefs_l2)
coefs_l1 = np.array(coefs_l1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), sharey=True)
for ax, coefs, title in [(axes[0], coefs_l2, "L2 — Ridge"),
                         (axes[1], coefs_l1, "L1 — Lasso")]:
    for i, col in enumerate(X_df.columns):
        ax.plot(alphas, coefs[:, i], lw=2)
        # Подписываем у ЛЕВОГО края: справа кривые сходятся к нулю
        # и подписи налезают друг на друга
        if abs(coefs[0, i]) > 5:
            ax.annotate(col, xy=(alphas[0], coefs[0, i]),
                        xytext=(-5, 0), textcoords="offset points",
                        fontsize=9, va="center", ha="right")
    ax.axhline(0, color="#6B7484", lw=1)
    ax.set_xscale("log")
    ax.set_xlim(alphas[0] / 2.2, alphas[-1] * 1.15)
    ax.set_xlabel("alpha")
    alive = int((np.abs(coefs[-1]) > 0.5).sum())
    ax.set_title(f"{title}: при максимальной alpha ненулевых весов "
                 f"{alive} из {coefs.shape[1]}")

axes[0].set_ylabel("вес признака")
plt.tight_layout()
plt.show()

print(f"норма весов почти без штрафа:    {np.linalg.norm(coefs_l2[0]):.1f}")
print(f"норма весов при максимальной L2: {np.linalg.norm(coefs_l2[-1]):.1f}")
print(f"норма весов при максимальной L1: {np.linalg.norm(coefs_l1[-1]):.1f}")

Видно ровно то, о чём говорилось на лекции: **L2 сжимает веса плавно и не обнуляет**, **L1 обнуляет их по одному**. При большой `alpha` от модели с L1 остаётся два-три признака — это и есть автоматический отбор.

In [ ]:
# Сколько признаков выживает при разной силе L1.
#
# Важная деталь: НАШ субградиентный спуск точных нулей не даёт. Вес,
# который должен занулиться, болтается около нуля с амплитудой ~lr*alpha,
# и проверка (w != 0) всегда вернёт True. Настоящие реализации используют
# proximal-шаг, который обрезает вес до нуля честно — так делает Lasso
# из scikit-learn, на нём и посмотрим отбор
from sklearn.linear_model import Lasso

alphas_l1 = np.logspace(-2, 1.7, 30)
nonzero, our_small = [], []
for a in alphas_l1:
    w_sk = Lasso(alpha=a, max_iter=20000).fit(X_train, y_train_np).coef_
    nonzero.append(int((w_sk != 0).sum()))

    # для сравнения: сколько весов наша реализация увела «почти в ноль»
    w_our = LinearRegressionGD(lr=0.05, n_epochs=300, penalty="l1",
                               alpha=a).fit(X_train, y_train_np).w
    our_small.append(int((np.abs(w_our) > max(0.5, 3 * 0.05 * a)).sum()))

plt.figure(figsize=(9, 3.8))
plt.step(alphas_l1, nonzero, where="post", lw=2.4, color="#2B5CE6",
         label="Lasso из sklearn: точные нули")
plt.plot(alphas_l1, our_small, lw=1.8, ls="--", color="#6B7484",
         label="наш спуск: весов заметно больше нуля")
plt.xscale("log")
plt.yticks(range(0, 11, 2))
plt.xlabel("alpha")
plt.ylabel("признаков осталось в модели")
plt.title("L1-регуляризация отбирает признаки")
plt.legend()
plt.tight_layout()
plt.show()

print(f"при alpha = {alphas_l1[0]:.2f} в модели {nonzero[0]} признаков, "
      f"при alpha = {alphas_l1[-1]:.0f} — уже {nonzero[-1]}")

### Задание 2

Подберите оптимальную силу регуляризации для `Ridge` и `Lasso`. Сетка `alpha`, график и поиск максимума уже готовы: **допишите две строки** внутри цикла — соберите пайплайн и оцените его на кросс-валидации.

Посмотрите на форму кривых: что происходит слева, что справа и насколько регуляризация обошла обычную регрессию.

> **Что посмотреть:** сетку значений удобно задать через `np.logspace(-3, 3, 40)` — она равномерна в логарифме, как и надо для `alpha`. Качество — `cross_val_score(pipe, X, y, cv=5, scoring="r2")`, лучшее значение находится через `np.argmax`. `Lasso` может ругаться на сходимость — увеличьте `max_iter`.

In [ ]:
# Регуляризации нечего чинить на десяти признаках — переобучаться не на чем.
# Расширим пространство полиномами: 10 признаков превратятся в 65,
# и модель начнёт переобучаться по-настоящему
from sklearn.preprocessing import PolynomialFeatures

alphas_grid = np.logspace(-2, 3.5, 30)
scores = {"Ridge": [], "Lasso": []}

for a in alphas_grid:
    for name, Model in [("Ridge", Ridge), ("Lasso", Lasso)]:

        # Полиномы и масштабирование обязаны быть внутри пайплайна:
        # иначе статистики посчитаются по всем фолдам сразу — это утечка
        pipe = Pipeline([
            ("poly", PolynomialFeatures(2, include_bias=False)),
            ("scale", StandardScaler()),
            ("model", Model(alpha=a, max_iter=20000)),
        ])
        mean_score = cross_val_score(pipe, X_train_df, y_train, cv=5,
                                     scoring="r2").mean()

        scores[name].append(mean_score)

baseline = cross_val_score(
    Pipeline([("poly", PolynomialFeatures(2, include_bias=False)),
              ("scale", StandardScaler()), ("model", LinearRegression())]),
    X_train_df, y_train, cv=5, scoring="r2",
).mean()

plt.figure(figsize=(10, 4.5))
for name, vals in scores.items():
    plt.plot(alphas_grid, vals, marker=".", label=name)
plt.axhline(baseline, color="#C0393B", linestyle="--",
            label=f"без регуляризации ({baseline:.3f})")
plt.xscale("log")
plt.xlabel("alpha")
plt.ylabel("R² на кросс-валидации")
plt.title("65 полиномиальных признаков: регуляризация решает")
plt.legend()
plt.show()

best_ridge_idx = int(np.argmax(scores["Ridge"]))
best_lasso_idx = int(np.argmax(scores["Lasso"]))
best_alpha_ridge = alphas_grid[best_ridge_idx]
best_r2_ridge = scores["Ridge"][best_ridge_idx]
best_alpha_lasso = alphas_grid[best_lasso_idx]
best_r2_lasso = scores["Lasso"][best_lasso_idx]

print(f"Ridge: лучший alpha = {best_alpha_ridge:.2f}, R² = {best_r2_ridge:.4f}")
print(f"Lasso: лучший alpha = {best_alpha_lasso:.2f}, R² = {best_r2_lasso:.4f}")
print(f"без регуляризации: R² = {baseline:.4f}")

print()
print("Выводы:")
print("  Слева регуляризации почти нет — модель переобучается на 65 признаках")
print("  и на новых данных работает хуже, чем на десяти исходных.")
print("  Справа штраф задавливает веса в ноль, и модель недообучается.")
print("  Оптимум посередине, и выигрыш над нерегуляризованной моделью")
print("  здесь уже существенный — именно так регуляризация и работает:")
print("  она нужна там, где признаков много, а объектов мало.")


In [ ]:
# Проверка задания 2
assert round(float(best_alpha_ridge), 2) == 96.11, f"best_alpha_ridge = {best_alpha_ridge}"
assert round(float(best_r2_ridge), 2) == 0.42, f"best_r2_ridge = {best_r2_ridge}"
assert round(float(best_alpha_lasso), 2) == 4.52, f"best_alpha_lasso = {best_alpha_lasso}"
assert round(float(best_r2_lasso), 2) == 0.44, f"best_r2_lasso = {best_r2_lasso}"
assert round(float(baseline), 2) == 0.30, f"baseline = {baseline}"
print("Задание 2 — верно ✓")

> **Для преподавателя.** Если у Lasso при больших alpha R² уходит в отрицательные значения — это правильно: все веса обнулены, модель предсказывает константу и работает хуже, чем предсказание средним.

---
## 5. Логистическая регрессия

Тот же аппарат, но для классификации. Реализуем и сравним.

**Задача.** По снимку опухоли молочной железы определить, доброкачественная она или злокачественная. Снимок уже обработан: с изображения сняты характеристики клеточных ядер, и модель работает с числами, а не с картинкой.

**Признаки.** Измеряется десять характеристик одного ядра:

| характеристика | что это |
|---|---|
| `radius`, `perimeter`, `area` | размер ядра |
| `texture` | разброс яркости внутри ядра |
| `smoothness` | насколько плавный контур |
| `compactness`, `concavity`, `concave points` | насколько контур изрезан вогнутостями |
| `symmetry`, `fractal dimension` | симметричность и «изломанность» границы |

Каждая из десяти посчитана в трёх видах, отсюда 30 столбцов:

- `mean ...` — среднее по всем ядрам снимка
- `... error` — разброс между ядрами
- `worst ...` — среднее по трём самым крупным ядрам

**Целевая переменная.** `1` — доброкачественная опухоль, `0` — злокачественная. Обратите внимание на порядок: положительный класс здесь «всё хорошо», а не «болезнь».

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

cancer = load_breast_cancer(as_frame=True)
Xc, yc = cancer.data, cancer.target

print(f"объектов: {Xc.shape[0]}, признаков: {Xc.shape[1]}")
print(f"доброкачественных (1): {(yc == 1).sum()}, "
      f"злокачественных (0): {(yc == 0).sum()}")
print()

# Смотрим на сами данные: четыре характеристики в трёх видах
show = ["mean radius", "radius error", "worst radius",
        "mean texture", "mean concave points", "worst concave points"]
display(Xc[show].join(yc.rename("класс")).head())

In [ ]:
# Как отличаются классы по паре самых говорящих признаков
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col in zip(axes, ["worst radius", "worst concave points"]):
    for cls, color, label in [(1, "#1B8A5A", "доброкачественная"),
                              (0, "#C0393B", "злокачественная")]:
        ax.hist(Xc.loc[yc == cls, col], bins=30, alpha=0.65,
                color=color, label=label)
    ax.set_xlabel(col)
    ax.set_ylabel("снимков")
    ax.legend()
axes[0].set_title("Крупные ядра — тревожный признак")
axes[1].set_title("Изрезанный контур — тоже")
plt.tight_layout()
plt.show()

print("Распределения заметно разъезжаются — значит линейная граница")
print("между классами существует, и логистической регрессии есть на что опереться.")

In [ ]:
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    Xc, yc, test_size=0.2, random_state=RANDOM_STATE, stratify=yc
)
sc = StandardScaler().fit(Xc_train)
Xc_train_s, Xc_test_s = sc.transform(Xc_train), sc.transform(Xc_test)

print(f"обучающая: {Xc_train.shape[0]}, тестовая: {Xc_test.shape[0]}")

In [ ]:
class LogisticRegressionGD:
    """Логистическая регрессия на градиентном спуске.

    Отличие от линейной — только в двух местах:
    предсказание пропускается через сигмоиду, а функция потерь логарифмическая.
    Формула градиента при этом получается ровно такой же по виду.
    """

    def __init__(self, lr=0.1, n_epochs=500, alpha=0.0, random_state=42):
        self.lr = lr
        self.n_epochs = n_epochs
        self.alpha = alpha
        self.random_state = random_state

    @staticmethod
    def _sigmoid(z):
        # Численно устойчивый вариант: без него exp переполняется
        return np.where(z >= 0,
                        1 / (1 + np.exp(-np.clip(z, -500, 500))),
                        np.exp(np.clip(z, -500, 500)) / (1 + np.exp(np.clip(z, -500, 500))))

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        self.history_ = [self._loss(X, y)]

        for _ in range(self.n_epochs):
            p = self._sigmoid(X @ self.w + self.b)
            error = p - y
            grad_w = X.T @ error / n + 2 * self.alpha * self.w
            grad_b = error.mean()
            self.w -= self.lr * grad_w
            self.b -= self.lr * grad_b

            self.history_.append(self._loss(X, y))
        return self

    def _loss(self, X, y, eps=1e-12):
        p = self._sigmoid(X @ self.w + self.b)
        return -np.mean(y * np.log(p + eps) + (1 - y) * np.log(1 - p + eps))

    def predict_proba(self, X):
        return self._sigmoid(X @ self.w + self.b)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)


log_gd = LogisticRegressionGD(lr=0.5, n_epochs=1000, alpha=0.001)
log_gd.fit(Xc_train_s, yc_train.to_numpy())

plt.figure(figsize=(9, 3.5))
plt.plot(log_gd.history_, color="#2B5CE6")
plt.xlabel("эпоха")
plt.ylabel("логистические потери")
plt.title("Обучение логистической регрессии")
plt.show()

In [ ]:
# Сравнение с scikit-learn
sk_log = LogisticRegression(max_iter=5000, C=1/(2*0.001*len(Xc_train_s)))
sk_log.fit(Xc_train_s, yc_train)

our_proba = log_gd.predict_proba(Xc_test_s)
sk_proba = sk_log.predict_proba(Xc_test_s)[:, 1]

print(f"{'':22} {'accuracy':>10} {'ROC-AUC':>10}")
print(f"{'наша реализация':22} "
      f"{accuracy_score(yc_test, log_gd.predict(Xc_test_s)):10.4f} "
      f"{roc_auc_score(yc_test, our_proba):10.4f}")
print(f"{'scikit-learn':22} "
      f"{accuracy_score(yc_test, sk_log.predict(Xc_test_s)):10.4f} "
      f"{roc_auc_score(yc_test, sk_proba):10.4f}")
print()
print(f"корреляция предсказанных вероятностей: "
      f"{np.corrcoef(our_proba, sk_proba)[0, 1]:.4f}")

### Задание 3

Изучите коэффициенты обученной модели. График, сравнение весов и подсчёт метрик уже написаны: **допишите две строки** — заверните веса в `Series` с именами признаков и обучите такую же модель на немасштабированных данных.

Дальше посмотрите на результат и объясните словами: что означает знак у самых сильных признаков и почему без масштабирования веса нельзя сравнивать между собой.

> **Что посмотреть:** веса удобно завернуть в `pd.Series(model.w, index=X.columns)` — тогда имена признаков не потеряются. Сортировка по модулю: `s.reindex(s.abs().sort_values(ascending=False).index)`, максимум по модулю — `s.abs().idxmax()`. График — `Series.plot(kind="barh")`.

In [ ]:
# 1. Веса в Series — так имена признаков остаются рядом со значениями
coefs = pd.Series(log_gd.w, index=Xc.columns)

# 2. Та же модель, но на сырых признаках
log_raw = LogisticRegressionGD(lr=0.5, n_epochs=1000, alpha=0.001)
log_raw.fit(Xc_train.to_numpy(), yc_train.to_numpy())

# Дальше всё готово: топ-10 по модулю веса и график
top = coefs.reindex(coefs.abs().sort_values(ascending=False).index)[:10]

plt.figure(figsize=(9, 4.5))
colors = ["#1B8A5A" if v > 0 else "#C0393B" for v in top.sort_values()]
top.sort_values().plot(kind="barh", color=colors)
plt.title("Десять самых влиятельных признаков")
plt.xlabel("вес (плюс — в пользу класса «доброкачественная»)")
plt.tight_layout()
plt.show()
print(top.round(3).to_string())

compare = pd.DataFrame({
    "масштабированные": log_gd.w,
    "сырые": log_raw.w,
}, index=Xc.columns)
print()
print("сравнение весов (первые 6 признаков):")
display(compare.head(6).round(4))

top_feature = top.abs().idxmax()
auc_scaled = roc_auc_score(yc_test, our_proba)
auc_raw = roc_auc_score(yc_test, log_raw.predict_proba(Xc_test.to_numpy()))

print(f"\nROC-AUC с масштабированием:  {auc_scaled:.4f}")
print(f"ROC-AUC без масштабирования: {auc_raw:.4f}")

print()
print("В этом датасете класс 1 — доброкачественная опухоль. Признаки")
print("с приставкой 'worst' описывают самые крупные значения измерений.")
print("Отрицательный вес у 'worst radius' означает: чем больше радиус,")
print("тем ниже вероятность доброкачественной опухоли.")
print()
print("Без масштабирования веса несопоставимы: 'mean area' измеряется")
print("сотнями, а 'mean smoothness' — сотыми долями. Сравнивать их")
print("коэффициенты бессмысленно, а регуляризация штрафует неравномерно.")
print("Качество тоже падает: спуск плохо сходится на вытянутой поверхности.")


In [ ]:
# Проверка задания 3
assert top_feature == "worst texture", f"top_feature = {top_feature}"
assert round(float(auc_scaled), 2) == 1.0, f"auc_scaled = {auc_scaled}"
assert round(float(auc_raw), 2) == 0.92, f"auc_raw = {auc_raw}"
assert auc_scaled > auc_raw, "с масштабированием качество должно быть выше"
print("Задание 3 — верно ✓")

> **Для преподавателя.** Это ключевой вывод практики: интерпретация коэффициентов возможна только после масштабирования. Студенты часто пытаются сравнивать веса на сырых данных и делают неверные выводы о важности признаков.

---
## Итоги занятия

- Градиентный спуск устроен просто: вычислить градиент, шагнуть против него, повторить
- Тот же алгоритм обучает нейросети — там меняется только способ вычисления градиента
- **Скорость обучения критична:** мала — не сойдётся, велика — разойдётся
- Мини-пакетный режим — практический компромисс между точностью и скоростью
- L2 сжимает веса, L1 обнуляет и отбирает признаки
- Логистическая регрессия отличается от линейной сигмоидой и функцией потерь
- **Интерпретировать коэффициенты можно только на отмасштабированных признаках**

## Домашнее задание 2

Выдаётся после лекции 4. Реализуете градиентный спуск самостоятельно и добавите к нему честную валидацию.